# 01 — Hello World (HTML)

Il più semplice esempio possibile con `HtmlBuilderHandler`.

**Cosa si impara qui:**
- Sottoclassare `HtmlBuilderHandler` e bindare un dialetto al handler (decisione 9).
- Popolare la `source` Bag dentro `main(self, root)` (decisione 5).
- Il ciclo a tre fasi esplicite: `create()` → `build()` → `render()`.
- Visualizzare la *ricetta* (`source.to_xml()`) e l'output finale (`render()`).
- Switchare tra `xml=True` (default, XML well-formed) e `xml=False` (HTML5 idiomatico).

## 1. Definire una pagina

Una pagina è una sottoclasse di `HtmlBuilderHandler` che implementa `main(self, root)`.
Il parametro `root` è la `source` Bag su cui si scrive la ricetta della pagina.

In [ ]:
from genro_builders.contrib.html import HtmlBuilderHandler


class HelloPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.h1("Hello World")
        body.p("Questa è la mia prima pagina con genro-builders.")

## 2. `create()` — popola la source

`create()` chiama `main(self.source)` e popola la source con la ricetta.
La source è la rappresentazione utente, preservata fedelmente (decisione 7).

In [ ]:
page = HelloPage()
page.create()

print(page.source.to_xml())

## 3. `build()` — materializza la source in `built`

`build()` produce la `built` Bag. Per il caso base senza `@component` o iterate è un mirror 1:1.
La distinzione tra source e built diventa importante quando i component vanno espansi (decisione 7).

In [ ]:
page.build()
print(page.built.to_xml())

## 4. `render()` — produce l'HTML finale

Il dialetto definisce *come* serializzare la built. `HtmlBuilder._default_render_mode` è `"html"`.

In [ ]:
print(page.render())

## 5. Preview inline (Jupyter)

In [ ]:
from IPython.display import HTML

HTML(page.render())

## 6. Void tag e stile self-close

I void tag HTML5 (`img`, `br`, `hr`, `input`, ...) per default sono emessi self-close XHTML-style: `<img src="x"/>`.
Così il documento è anche XML well-formed (utile in pipeline che mescolano HTML e SVG/XML).
Con `xml=False` si ottiene la forma HTML5 idiomatica: `<img src="x">`.

In [ ]:
class LogoPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.img(src="logo.png", alt="Logo")
        body.br()
        body.p("Sotto il logo.")


logo = LogoPage()
logo.create()
logo.build()

print("xml=True  (default):", logo.render())
print("xml=False (HTML5):  ", logo.render(xml=False))

## 7. Booleani come letterali JS

Gli attributi `True`/`False` vengono serializzati come stringhe `"true"`/`"false"`,
così il JS lato client può consumarli direttamente.
Il valore `None` viene oggi filtrato a monte (vedi `tests/test_html_render.py`).

In [ ]:
class FormPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.input(type="text", disabled=True)
        body.input(type="checkbox", checked=False)


form = FormPage()
form.create()
form.build()
print(form.render())

## 8. Attributi keyword-collision: `_class` → `class`

Python non accetta `class` come keyword arg, quindi si scrive `_class`. Il renderer mappa l'underscore-prefix verso il nome HTML corretto (idem per `_for`).

In [ ]:
class StyledPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.div("Importante", _class="alert primary")


styled = StyledPage()
styled.create()
styled.build()
print(styled.render())